# MANZAR — Fast Sparse-to-Mesh 3D Pipeline

**No CUDA dense stereo needed.** Runs on CPU + optional GPU in ~2-3 minutes.

Steps:
1. Add `manzar-photos` dataset as input.
2. Run the single code cell below.

It will:
- Install COLMAP (CPU sparse)
- Install Open3D
- Run COLMAP sparse reconstruction
- Build a clean mesh from the sparse point cloud using Open3D Poisson
- Save outputs to `/kaggle/working/manzar_outputs/`

In [1]:
# MANZAR — Fast Sparse-to-Mesh 3D Pipeline# No CUDA dense stereo needed. Runs in ~2-3 minutes.
"""
MANZAR — Fast Sparse-to-Mesh 3D Reconstruction for Kaggle / Colab
No CUDA dense stereo needed. Uses COLMAP sparse + Open3D Poisson mesh.
"""

from __future__ import annotations

import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

os.environ["QT_QPA_PLATFORM"] = "offscreen"

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
WORKSPACE = Path("/kaggle/working/manzar_workspace")
OUTPUT_DIR = Path("/kaggle/working/manzar_outputs")


def find_input_dir() -> Path | None:
    """Find a folder containing JPG/PNG photos under /kaggle/input or /content."""
    for base in [Path("/kaggle/input"), Path("/content")]:
        if not base.exists():
            continue
        candidates = []
        for p in base.rglob("*"):
            if p.is_dir():
                imgs = (
                    list(p.glob("*.jpg")) + list(p.glob("*.JPG")) +
                    list(p.glob("*.jpeg")) + list(p.glob("*.JPEG")) +
                    list(p.glob("*.png")) + list(p.glob("*.PNG"))
                )
                if imgs:
                    candidates.append((len(imgs), p))
        if candidates:
            candidates.sort(reverse=True)
            return candidates[0][1]
    return None


INPUT_DIR = find_input_dir()
COLMAP = "colmap"


def run(cmd: list[str], stage: str, timeout: int = 3600) -> None:
    """Run a command with logging."""
    print(f"\n[{stage}] {' '.join(cmd)}")
    start = time.time()
    env = os.environ.copy()
    env["QT_QPA_PLATFORM"] = "offscreen"
    proc = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        timeout=timeout,
        env=env,
    )
    elapsed = time.time() - start
    print(f"  -> exit {proc.returncode} in {elapsed:.1f}s")
    if proc.stdout:
        for line in proc.stdout.strip().splitlines()[-20:]:
            print("    " + line)
    if proc.returncode != 0:
        raise RuntimeError(f"{stage} failed (exit {proc.returncode})")


def install_colmap() -> None:
    """Install COLMAP via apt (CPU-only is fine for sparse)."""
    print("\n[Install] COLMAP via apt...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "colmap"], check=True)
    result = subprocess.run([COLMAP, "--help"], capture_output=True, text=True,
                            env={**os.environ, "QT_QPA_PLATFORM": "offscreen"})
    print("  ", result.stdout.splitlines()[0] if result.stdout else result.stderr)


def install_open3d() -> None:
    """Install Open3D via pip."""
    print("\n[Install] Open3D via pip...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open3d"], check=True)
    import open3d as o3d
    print(f"  Open3D version: {o3d.__version__}")


def check_gpu() -> bool:
    """Check if CUDA GPU is available."""
    print("\n[GPU Check]")
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(result.stdout or result.stderr)
    return result.returncode == 0


def mesh_from_sparse(ply_path: Path, out_dir: Path) -> None:
    """Create a clean mesh from sparse point cloud using Open3D."""
    import open3d as o3d
    import numpy as np

    print("\n[Open3D] Loading sparse point cloud...")
    pcd = o3d.io.read_point_cloud(str(ply_path))
    print(f"  Points before filtering: {len(pcd.points)}")

    pts = np.asarray(pcd.points)
    bbox = pts.max(axis=0) - pts.min(axis=0)
    print(f"  Bounding box size: {bbox}")

    # Use a radius relative to the point cloud scale for outlier removal
    radius = float(np.percentile(bbox, 50) * 0.02)
    if radius <= 0 or not np.isfinite(radius):
        radius = 0.05

    if len(pcd.points) >= 100:
        # Mild statistical outlier removal
        pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=3.0)
        # Radius outlier only if it leaves enough points
        pcd_tmp, _ = pcd.remove_radius_outlier(nb_points=8, radius=radius)
        if len(pcd_tmp.points) >= max(100, int(0.3 * len(pcd.points))):
            pcd = pcd_tmp
            print(f"  Points after filtering: {len(pcd.points)}")
        else:
            print(f"  Skipping radius filter (would remove too many points).")
    else:
        print("  Skipping filtering (too few points).")

    if len(pcd.points) == 0:
        print("\n❌ No points left after filtering. Saving original point cloud only.")
        return

    # Estimate normals with adaptive radius
    normal_radius = float(np.percentile(bbox, 50) * 0.05)
    if normal_radius <= 0 or not np.isfinite(normal_radius):
        normal_radius = 0.1
    print(f"  Normal estimation radius: {normal_radius:.4f}")

    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(
            radius=normal_radius, max_nn=30
        )
    )
    pcd.orient_normals_consistent_tangent_plane(min(100, len(pcd.points) - 1))

    # Poisson reconstruction
    print("\n[Open3D] Running Poisson reconstruction...")
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        pcd, depth=9
    )

    # Trim low-density regions
    densities = np.asarray(densities)
    density_mask = densities > np.quantile(densities, 0.1)
    mesh.remove_vertices_by_mask(~density_mask)

    # Clean mesh
    mesh.remove_degenerate_triangles()
    mesh.remove_non_manifold_edges()
    mesh.remove_duplicated_triangles()
    mesh.remove_duplicated_vertices()

    mesh_path = out_dir / "mesh_sparse.obj"
    mesh_path_ply = out_dir / "mesh_sparse.ply"
    o3d.io.write_triangle_mesh(str(mesh_path), mesh)
    o3d.io.write_triangle_mesh(str(mesh_path_ply), mesh)
    print(f"  Mesh saved: {mesh_path}")
    print(f"  Vertices: {len(mesh.vertices)}, Triangles: {len(mesh.triangles)}")


def main() -> int:
    print("=" * 70)
    print("MANZAR — Fast Sparse-to-Mesh 3D Reconstruction")
    print("No CUDA dense stereo needed.")
    print("=" * 70)

    check_gpu()
    install_colmap()
    install_open3d()

    if INPUT_DIR is None or not INPUT_DIR.exists() or not list(INPUT_DIR.iterdir()):
        print("\n❌ No photos found. Upload JPG/PNG photos and re-run.")
        return 1

    photos = sorted([p for p in INPUT_DIR.iterdir()
                     if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
    print(f"\n📸 Found {len(photos)} photos in {INPUT_DIR}")

    # Clean workspace
    if WORKSPACE.exists():
        shutil.rmtree(WORKSPACE)
    WORKSPACE.mkdir(parents=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    images_dir = WORKSPACE / "images"
    images_dir.mkdir(parents=True, exist_ok=True)
    for p in photos:
        shutil.copy(p, images_dir / p.name)

    db_path = WORKSPACE / "database.db"
    sparse_dir = WORKSPACE / "sparse"
    sparse_dir.mkdir(parents=True, exist_ok=True)

    # 1. Feature extraction (CPU)
    run(
        [
            COLMAP, "feature_extractor",
            "--database_path", str(db_path),
            "--image_path", str(images_dir),
            "--ImageReader.single_camera", "1",
            "--SiftExtraction.max_num_features", "8192",
            "--SiftExtraction.use_gpu", "0",
        ],
        stage="Feature Extraction",
    )

    # 2. Feature matching (CPU)
    run(
        [COLMAP, "exhaustive_matcher",
         "--database_path", str(db_path),
         "--SiftMatching.use_gpu", "0"],
        stage="Feature Matching",
    )

    # 3. Sparse reconstruction
    run(
        [COLMAP, "mapper",
         "--database_path", str(db_path),
         "--image_path", str(images_dir),
         "--output_path", str(sparse_dir)],
        stage="Sparse Reconstruction",
    )

    # Find best sparse model
    sparse_model_dir = None
    for d in sorted(sparse_dir.iterdir()):
        if d.is_dir() and (d / "images.bin").exists():
            sparse_model_dir = d
            break

    if sparse_model_dir is None:
        print("\n❌ Sparse reconstruction failed: no model found.")
        return 1

    print(f"\n✅ Sparse model: {sparse_model_dir}")

    # Export sparse point cloud
    sparse_ply = OUTPUT_DIR / "sparse_pointcloud.ply"
    run(
        [COLMAP, "model_converter",
         "--input_path", str(sparse_model_dir),
         "--output_path", str(sparse_ply),
         "--output_type", "PLY"],
        stage="Export Sparse PLY",
    )

    # 4. Create mesh from sparse point cloud
    mesh_from_sparse(sparse_ply, OUTPUT_DIR)

    # Package outputs
    print("\n[Package] Zipping outputs...")
    zip_path = OUTPUT_DIR / "manzar_outputs.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in OUTPUT_DIR.iterdir():
            if f.name == "manzar_outputs.zip":
                continue
            zf.write(f, arcname=f.name)

    print(f"\n🎉 Done! Outputs in {OUTPUT_DIR}:")
    for f in sorted(OUTPUT_DIR.iterdir()):
        size = f.stat().st_size / (1024 * 1024)
        print(f"   {f.name:<30} {size:>8.2f} MB")

    return 0


if __name__ == "__main__":
    raise SystemExit(main())

main()

MANZAR — Fast Sparse-to-Mesh 3D Reconstruction
No CUDA dense stereo needed.

[GPU Check]
Thu Sep  3 07:40:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             25W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |        

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


Selecting previously unselected package libdouble-conversion3:amd64.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../00-libdouble-conversion3_3.1.7-4_amd64.deb ...
Unpacking libdouble-conversion3:amd64 (3.1.7-4) ...
Selecting previously unselected package libqt5core5a:amd64.
Preparing to unpack .../01-libqt5core5a_5.15.3+dfsg-2ubuntu0.2_amd64.deb ...
Unpacking libqt5core5a:amd64 (5.15.3+dfsg-2ubuntu0.2) ...
Selecting previously unselected package libevdev2:amd64.
Preparing to unpack .../02-libevdev2_1.12.1+dfsg-1_amd64.deb ...
Unpacking libevdev2:amd64 (1.12.1+dfsg-1) ...
Selecting previously unselected package libmtdev1:amd64.
Preparing to unpack .../03-libmtdev1_1.1.6-1build4_amd64.deb ...
Unpacking libmtdev1:amd64 (1.1.6-1build4) ...
Selecting previously unselected package libgudev-1.0-0:amd64.
Preparing to unpack .../04-libgudev-1.0-0_1%3a237-2build1_amd64.deb ...
Unpacking libgudev-1.0-0:amd64 (1:237-2build1) ...
Selecting previousl

SystemExit: 0

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [9]:
# List outputs
!ls -lh /kaggle/working/manzar_outputs/

total 7.9M
-rw-r--r-- 1 root root 1.7M Aug 29 20:47 manzar_outputs.zip
-rw-r--r-- 1 root root 4.2M Aug 29 20:47 mesh_sparse.obj
-rw-r--r-- 1 root root 2.0M Aug 29 20:47 mesh_sparse.ply
-rw-r--r-- 1 root root 119K Aug 29 20:47 sparse_pointcloud.ply
